<a href="https://colab.research.google.com/github/fayashi06/Deep-Learning-Classification-/blob/main/Diamonds_dataset_new2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader


In [ ]:
!curl -L -o diamonds.zip\
  https://www.kaggle.com/api/v1/datasets/download/shivam2503/diamonds

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  733k  100  733k    0     0  1134k      0 --:--:-- --:--:-- --:--:-- 1134k


In [ ]:

!unzip diamonds.zip

Archive:  diamonds.zip
  inflating: diamonds.csv            


In [ ]:
df = pd.read_csv('diamonds.csv')

In [ ]:
df.isnull().sum()

,0
Unnamed: 0,0
carat,0
cut,0
color,0
clarity,0
depth,0
table,0
price,0
x,0
y,0


In [ ]:
df

,Unnamed: 0,carat,cut,color,clarity,depth,table,price,x,y,z
0,1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75
...,...,...,...,...,...,...,...,...,...,...,...
53935,53936,0.72,Ideal,D,SI1,60.8,57.0,2757,5.75,5.76,3.50
53936,53937,0.72,Good,D,SI1,63.1,55.0,2757,5.69,5.75,3.61
53937,53938,0.70,Very Good,D,SI1,62.8,60.0,2757,5.66,5.68,3.56
53938,53939,0.86,Premium,H,SI2,61.0,58.0,2757,6.15,6.12,3.74


In [ ]:
X = df.drop('price', axis=1)
y = df['price'].copy()

In [ ]:
X = pd.get_dummies(X, drop_first=True)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X_train_full,y_train_full,test_size=0.2, random_state=42)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

In [ ]:
X_train.dtype

dtype('float64')

In [ ]:
X_train.shape

(34521, 24)

In [ ]:
X_test.shape

(10788, 24)

In [ ]:
X_valid.shape

(8631, 24)

In [ ]:
X_train.dtype

dtype('float64')

In [ ]:
#CONVERTING TO TENSOR DATA

In [ ]:
if torch.cuda.is_available():
  device='cuda'
elif torch.backends.mps.is_available():
  device='mps'
else:
  device='cpu'

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test=torch.tensor(X_test, dtype=torch.float32)
X_valid=torch.tensor(X_valid, dtype=torch.float32)

In [ ]:
X_train = X_train.to(device)
X_valid = X_valid.to(device)
X_test  = X_test.to(device)

In [ ]:
X_train.device

device(type='cpu')

In [ ]:
X_train.dtype

torch.float32

In [ ]:
X_test.dtype

torch.float32

In [ ]:
X_valid.dtype

torch.float32

In [ ]:
means = X_train.mean(dim=0,keepdim=True) #datani normallasdirmaq ucun istifade eledim
std = X_train.std(dim=0, keepdim=True)


std[std == 0] = 1

X_train = (X_train - means) / std
X_valid = (X_valid - means) / std
X_test  = (X_test - means) / std

In [ ]:
X_train.shape #torchda olub olmadigini yoxladim

torch.Size([34521, 24])

In [ ]:
y_train.dtype #kecirik y-e

dtype('int64')

In [ ]:
y_train.shape

(34521,)

In [ ]:
y_train = torch.tensor(y_train.to_numpy(), dtype=torch.float32).view(-1,1)
y_valid = torch.tensor(y_valid.to_numpy(), dtype=torch.float32).view(-1,1)
y_test  = torch.tensor(y_test.to_numpy(), dtype=torch.float32).view(-1,1)

In [ ]:
y_train.shape

torch.Size([34521, 1])

In [ ]:
y_test.shape

torch.Size([10788, 1])

In [ ]:
y_valid.shape

torch.Size([8631, 1])

In [ ]:
X_train.shape[1]

24

In [ ]:
from torch.utils.data import TensorDataset #tensora yukleyirik

In [ ]:
train_dataset = TensorDataset(X_train, y_train)
val_dataset   = TensorDataset(X_valid, y_valid)
test_dataset  = TensorDataset(X_test, y_test)

In [ ]:
from torch.utils.data import DataLoader #loadere yukleyirik

In [ ]:
train_loader = DataLoader(train_dataset,batch_size = 32, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 32, shuffle = False)
val_loader = DataLoader(val_dataset, batch_size = 32, shuffle = False)

In [ ]:
import torch.nn as nn

In [ ]:
class RegressionModel(nn.Module): #model qurdum(simple Regression modelidir)
  def __init__(self,input_dim):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(input_dim,128),
        nn.ReLU(),
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Linear(64,1)
    ).to(device)
  def forward(self,x):
    return self.net(x)

model = RegressionModel(X_train.shape[1]).to(device)

In [ ]:
model

RegressionModel(
  (net): Sequential(
    (0): Linear(in_features=24, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [ ]:
learning_rate = 0.0001
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
loss = nn.MSELoss()

In [ ]:
def mb_train(model, optimizer, criterion, train_loader, val_loader, n_epochs):
    train_losses = []
    valid_losses = []
    train_rmses = []
    valid_rmses = []

    for epoch in range(n_epochs):
        model.train()
        total_train_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_train_loss += loss.item()

        mean_train_loss = total_train_loss / len(train_loader)
        train_losses.append(mean_train_loss)
        train_rmses.append(torch.sqrt(torch.tensor(mean_train_loss)).item())

        model.eval()
        total_valid_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                y_pred = model(X_batch)
                loss = criterion(y_pred, y_batch)
                total_valid_loss += loss.item()

        mean_valid_loss = total_valid_loss / len(val_loader)
        valid_losses.append(mean_valid_loss)
        valid_rmses.append(torch.sqrt(torch.tensor(mean_valid_loss)).item())

        print(f"Epoch {epoch+1}/{n_epochs} | train_loss={mean_train_loss:.4f} | train_rmse={train_rmses[-1]:.4f} | valid_loss={mean_valid_loss:.4f} | valid_rmse={valid_rmses[-1]:.4f}")

    return train_losses, valid_losses, train_rmses, valid_rmses

In [ ]:
train_losses, valid_losses, train_rmses, valid_rmses = mb_train(
    model=model,
    optimizer=optimizer,
    criterion=loss,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=50
)

Epoch 1/50 | train_loss=985641.4972 | train_rmse=992.7948 | valid_loss=404272.4123 | valid_rmse=635.8242
Epoch 2/50 | train_loss=989755.8540 | train_rmse=994.8647 | valid_loss=402455.8965 | valid_rmse=634.3941
Epoch 3/50 | train_loss=991286.4115 | train_rmse=995.6337 | valid_loss=402125.9544 | valid_rmse=634.1340
Epoch 4/50 | train_loss=988920.8228 | train_rmse=994.4450 | valid_loss=402927.1808 | valid_rmse=634.7654
Epoch 5/50 | train_loss=988809.5549 | train_rmse=994.3890 | valid_loss=398790.5261 | valid_rmse=631.4987
Epoch 6/50 | train_loss=987473.0789 | train_rmse=993.7168 | valid_loss=401087.7338 | valid_rmse=633.3149
Epoch 7/50 | train_loss=988207.4638 | train_rmse=994.0862 | valid_loss=395554.2876 | valid_rmse=628.9311
Epoch 8/50 | train_loss=994697.7706 | train_rmse=997.3453 | valid_loss=396366.8080 | valid_rmse=629.5767
Epoch 9/50 | train_loss=990743.9524 | train_rmse=995.3612 | valid_loss=403583.0461 | valid_rmse=635.2819
Epoch 10/50 | train_loss=989652.3066 | train_rmse=994.8

In [ ]:
import numpy as np

print(np.isnan(X_train).any())
print(np.isnan(y_train).any())
print(np.isinf(X_train).any())
print(np.isinf(y_train).any())

tensor(0, dtype=torch.uint8)
tensor(0, dtype=torch.uint8)
tensor(0, dtype=torch.uint8)
tensor(0, dtype=torch.uint8)


/tmp/ipykernel_25291/2739461715.py:3: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  print(np.isnan(X_train).any())
/tmp/ipykernel_25291/2739461715.py:4: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  print(np.isnan(y_train).any())
/tmp/ipykernel_25291/2739461715.py:5: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  print(np.isinf(X_train).any())
/tmp/ipykernel_25291/2739461715.py:6: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  print(np.isinf(y_train).any())


In [ ]:
y_std = y_train.std()
print(y_std)

tensor(3994.9856)


In [ ]:
print(y_train.std())

tensor(3994.9856)


In [ ]:
X_batch, _ = next(iter(test_loader))
X_batch = X_batch.to(device)
y_pred = model(X_batch)

print("y_pred min/max:", y_pred.min().item(), y_pred.max().item())
print("y_pred has NaN:", torch.isnan(y_pred).any())
print("y_pred has Inf:", torch.isinf(y_pred).any())

y_pred min/max: 586.353515625 16069.8837890625
y_pred has NaN: tensor(False)
y_pred has Inf: tensor(False)


In [ ]:
with torch.no_grad():
    y_pred = model(X_train[:10])
    print(y_pred)

tensor([[5174.0137],
        [6937.0859],
        [3867.7715],
        [ 780.3701],
        [ 846.5631],
        [4154.6133],
        [6128.5605],
        [2925.3352],
        [ 963.0773],
        [ 725.2967]])


In [ ]:
df

,Unnamed: 0,carat,cut,color,clarity,depth,table,price,x,y,z
0,1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75
...,...,...,...,...,...,...,...,...,...,...,...
53935,53936,0.72,Ideal,D,SI1,60.8,57.0,2757,5.75,5.76,3.50
53936,53937,0.72,Good,D,SI1,63.1,55.0,2757,5.69,5.75,3.61
53937,53938,0.70,Very Good,D,SI1,62.8,60.0,2757,5.66,5.68,3.56
53938,53939,0.86,Premium,H,SI2,61.0,58.0,2757,6.15,6.12,3.74
